<a href="https://colab.research.google.com/github/medha-3102/Android-Me/blob/main/rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
!pip install -q sentence-transformers chromadb pypdf transformers accelerate

In [33]:
from sentence_transformers import SentenceTransformer
import chromadb

model = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.Client()
collection = client.get_or_create_collection("docs")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [121]:
import re

def chunk_text(text, max_chars=1000, overlap=200):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    current = ""
    for s in sentences:
        if len(current) + len(s) <= max_chars:
            current += " " + s
        else:
            chunks.append(current.strip())
            current = s[-overlap:] if overlap else s
    if current:
        chunks.append(current.strip())
    return chunks


In [ ]:
from google.colab import files
uploaded = files.upload()

from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
import chromadb

model = SentenceTransformer("all-MiniLM-L6-v2")

chroma = chromadb.Client()

try:
    chroma.delete_collection("docs")
except:
    pass

collection = chroma.create_collection("docs")

doc_id = 0


for file_name in uploaded.keys():
    reader = PdfReader(file_name)

    text = "\n".join(
        page.extract_text() or ""
        for page in reader.pages
    )

    chunks = chunk_text(text, max_chars=250, overlap=50)
    for chunk in chunks:
        collection.add(
            ids=[f"{file_name}_{doc_id}"],
            documents=[chunk],
            embeddings=[model.encode(chunk).tolist()],
            metadatas=[{"source": file_name}]
        )
        doc_id += 1

In [ ]:
from google.colab import files
uploaded = files.upload()

from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
import chromadb

model = SentenceTransformer("all-MiniLM-L6-v2")

chroma = chromadb.Client()

try:
    chroma.delete_collection("docs")
except:
    pass

collection = chroma.create_collection("docs")

doc_id = 0


for file_name in uploaded.keys():
    reader = PdfReader(file_name)

    text = "\n".join(
        page.extract_text() or ""
        for page in reader.pages
    )

    chunks = chunk_text(text, max_chars=250, overlap=50)
    for chunk in chunks:
        collection.add(
            ids=[f"{file_name}_{doc_id}"],
            documents=[chunk],
            embeddings=[model.encode(chunk).tolist()],
            metadatas=[{"source": file_name}]
        )
        doc_id += 1

In [ ]:
from google.colab import files
uploaded = files.upload()

from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
import chromadb

model = SentenceTransformer("all-MiniLM-L6-v2")

chroma = chromadb.Client()

try:
    chroma.delete_collection("docs")
except:
    pass

collection = chroma.create_collection("docs")

doc_id = 0


for file_name in uploaded.keys():
    reader = PdfReader(file_name)

    text = "\n".join(
        page.extract_text() or ""
        for page in reader.pages
    )

    chunks = chunk_text(text, max_chars=1000, overlap=200)
    for chunk in chunks:
        collection.add(
            ids=[f"{file_name}_{doc_id}"],
            documents=[chunk],
            embeddings=[model.encode(chunk).tolist()],
            metadatas=[{"source": file_name}]
        )
        doc_id += 1

In [113]:
import numpy as np

def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    if np.linalg.norm(a) == 0 or np.linalg.norm(b) == 0:
        return 0.0
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def ask(query, top_k_chunks=5, sentence_sim_threshold=0.6, fallback="I couldn’t find a clear answer in the documents."):
    """
    Retrieval-based, extractive QA using sentence-level semantic similarity.
    Returns answer only if context supports it; otherwise fallback.
    """
    global model, collection

    if not query or len(query.strip().split()) < 3:
        return "Please ask a complete question."

    q_emb = model.encode(query, normalize_embeddings=True).tolist()

    results = collection.query(query_embeddings=[q_emb], n_results=10)
    if not results or not results.get("documents") or not results["documents"][0]:
        return fallback

    context_chunks = results["documents"][0]

    chunk_sims = []
    for chunk in context_chunks:
        chunk_emb = model.encode(chunk, normalize_embeddings=True).tolist()
        sim = cosine_similarity(q_emb, chunk_emb)
        chunk_sims.append((sim, chunk))

    chunk_sims.sort(reverse=True, key=lambda x: x[0])
    top_chunks = [chunk for sim, chunk in chunk_sims[:top_k_chunks]]

    answer_sentences = []
    for chunk in top_chunks:
        sentences = [s.strip() for s in chunk.replace("\n", " ").split('.') if s.strip()]
        for sentence in sentences:
            sent_emb = model.encode(sentence, normalize_embeddings=True).tolist()
            sim = cosine_similarity(q_emb, sent_emb)
            if sim >= sentence_sim_threshold:
                answer_sentences.append(sentence)

    if not answer_sentences:
        return fallback

    full_answer = '. '.join(answer_sentences)
    if not full_answer.endswith('.'):
        full_answer += '.'

    return full_answer

In [107]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    device=-1
)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

In [120]:
print(ask("What happens on Day 1 Onboarding?"))
print(ask(" What tasks are performed during the Pre-Onboarding Process?"))
# What does the Training and Induction phase include?
# How long is the probation period, and what does it involve?
# What happens after a successful probation period?

Day 1 Onboarding Orientation, company policy briefing, system access, and welcome session.
Employee Onboarding Process – Internal Documentation Introduction This document describes the standard onboarding process for new employees joining the organization.
